# 04 · Pulse shaping and matched filtering

A symbol stream is a train of impulses; the **pulse** turns it into a
band-limited waveform.  The receiver filters with the **matched filter**
and samples once per symbol.  Two requirements drive the design:

* **bandwidth**: the spectrum must fit the channel;
* **no inter-symbol interference (ISI)**: at each sampling instant, the
  neighbours' pulses must contribute nothing.

This notebook covers:

1. the pulse objects and their spectra;
2. the Nyquist criterion and the raised-cosine family;
3. eye diagrams;
4. why the matched filter maximizes the SNR, and the root-raised-cosine
   split between transmitter and receiver;
5. sampling phase, roll-off and timing sensitivity;
6. how long the pulse must be (`span`).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import commkit as ck
from commkit import RC, RRC, Constellation, Gaussian, Rect
from commkit.backend import to_device

DEVICE = "gpu" if ck.backend.is_cupy_available() else "cpu"
print(DEVICE)

ck.plotting.apply_default_theme()
C16 = Constellation.qam(16)
SPS = 8


def host(a):
    return to_device(a, "cpu")

## 1. Pulses are objects

`RRC(rolloff=...)`, `RC(...)`, `Gaussian(fwhm=...)`, `Rect(...)` and
`SmoothRect(...)` describe a pulse; `pulse.taps(sps)` returns its samples
at a given oversampling.  Functions take the object (or a raw taps array),
and a `Signal` remembers the pulse it was shaped with.

A rectangular pulse is perfectly confined in time and therefore spreads in
frequency (a sinc, side lobes falling only 6 dB per octave).  Smooth pulses
trade time extent for a compact spectrum.

In [ ]:
pulses = {
    "Rect": Rect(),
    "Gaussian, FWHM 1 T": Gaussian(fwhm=1.0),
    "RC, β = 0.2": RC(rolloff=0.2),
    "RRC, β = 0.2": RRC(rolloff=0.2),
}
fig, (ax_t, ax_f) = plt.subplots(1, 2, figsize=(12, 4))
nfft = 1 << 14
for name, p in pulses.items():
    h = host(p.taps(SPS))
    t = (np.arange(h.size) - (h.size - 1) / 2) / SPS
    ax_t.plot(t, h / np.max(np.abs(h)), label=name)
    H = np.fft.fftshift(np.abs(np.fft.fft(h, nfft)))
    f = np.fft.fftshift(np.fft.fftfreq(nfft, 1 / SPS))
    ax_f.plot(f, 20 * np.log10(H / H.max() + 1e-12), label=name)
ax_t.set_xlim(-4, 4)
ax_t.set_xlabel("Time [symbol periods]")
ax_t.set_title("Pulse (peak-normalized)")
ax_f.set_xlim(0, 3)
ax_f.set_ylim(-80, 3)
ax_f.set_xlabel("Frequency [symbol rates]")
ax_f.set_ylabel("|P(f)| [dB]")
ax_f.set_title("Spectrum")
ax_t.legend()
plt.show()

## 2. The Nyquist criterion

A pulse `p(t)` causes no ISI when it is zero at every non-zero multiple of
the symbol period, `p(kT) = 0` for `k ≠ 0`.  In frequency, the folded
spectrum must be flat: `Σ_m P(f - m/T) = const`.  The narrowest such
spectrum is the ideal brick wall of width `1/T` (a sinc pulse, infinitely
long).  The **raised cosine** adds a cosine roll-off of relative width `β`:
the bandwidth grows to `(1 + β)/T`, and in return the pulse tails decay
as `1/t³`, which makes truncation and timing errors harmless.

Check the zero crossings of `RC` taps at the symbol instants:

In [ ]:
for beta in (0.05, 0.2, 0.5):
    h = host(RC(rolloff=beta).taps(SPS))
    centre = (h.size - 1) // 2
    at_symbols = h[centre % SPS :: SPS] / h[centre]
    worst = np.max(np.abs(np.delete(at_symbols, centre // SPS)))
    print(
        f"RC β = {beta:4.2f}: max |p(kT)/p(0)| over k != 0 is {worst:.1e}, "
        f"bandwidth {(1 + beta):.2f} / T"
    )

## 3. Eye diagrams

Overlaying the waveform over two symbol periods, triggered at the symbol
clock, gives the **eye diagram**.  An open eye at the sampling instant means
no ISI; the opening's height is the noise margin and its width the timing
margin.  The eye of an RC-shaped signal is open at the centre; an
RRC-shaped signal on its own is *not* ISI-free, which is the point of the
next section.

In [ ]:
tx_rc = ck.generate(
    Constellation.psk(4),
    num_symbols=4000,
    symbol_rate=1e9,
    sps=SPS,
    pulse=RC(rolloff=0.2),
    rng=1,
    device=DEVICE,
)
tx_rrc = ck.generate(
    Constellation.psk(4),
    num_symbols=4000,
    symbol_rate=1e9,
    sps=SPS,
    pulse=RRC(rolloff=0.2),
    rng=1,
    device=DEVICE,
)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ck.plotting.plot_eye_diagram(
    tx_rc.samples.real,
    sps=SPS,
    ax=axes[0],
    kind="line",
    title="QPSK, RC β = 0.2 (in-phase)",
)
ck.plotting.plot_eye_diagram(
    tx_rrc.samples.real,
    sps=SPS,
    ax=axes[1],
    kind="line",
    title="QPSK, RRC β = 0.2 (in-phase)",
)
plt.show()

## 4. The matched filter

In white Gaussian noise, the receive filter that maximizes the SNR at the
sampling instant is the **matched filter**, `h(t) = p*(-t)`: it collects
all the pulse energy while weighting the noise least.  The combined
response is `p(t) * p*(-t)`, so to have both the optimal SNR *and* zero ISI,
the pulse's energy spectrum `|P(f)|²` must be a Nyquist (raised-cosine)
spectrum.  That is the **root raised cosine**: RRC at the transmitter and
RRC at the receiver multiply to RC.

`matched_filter` uses the `Signal`'s pulse by default.  Compare the SNR at
the symbol instants for three receivers at Es/N0 = 15 dB:

In [ ]:
tx = ck.generate(
    C16,
    num_symbols=2**15,
    symbol_rate=1e9,
    sps=SPS,
    pulse=RRC(rolloff=0.2),
    rng=2,
    device=DEVICE,
)
rx = ck.impairments.apply_awgn(tx, esn0_db=15, rng=3)

receivers = {
    "no filter, sample directly": rx,
    "low-pass FIR at 0.6 Rs": ck.filtering.fir_filter(
        rx,
        ck.filtering.fir_taps(
            sampling_rate=rx.sampling_rate, num_taps=8 * SPS + 1, cutoff=0.6e9
        ),
    ),
    "matched (RRC)": ck.filtering.matched_filter(rx),
}
for name, y in receivers.items():
    z = ck.multirate.decimate_to_symbol_rate(y)
    # A real gain does not change the SNR; snr() divides it out against the reference.
    print(f"{name:28s} SNR = {ck.metrics.snr(z):5.2f} dB")

ck.plotting.plot_eye_diagram(
    ck.filtering.matched_filter(tx).samples.real,
    sps=SPS,
    kind="line",
    title="16-QAM after RRC transmit + RRC matched filter (noise-free)",
    show=True,
)

Without a filter the receiver samples the noise of the whole simulated band
(8 times the symbol rate), so it loses about `10·log10(8) = 9` dB.  The
low-pass filter removes most of that noise but leaves ISI, because
`RRC × low-pass` is not a Nyquist pulse.  The matched filter reaches the
Es/N0 of the channel.

## 5. Sampling phase and roll-off

After the matched filter the eye is open at one phase only.
`decimate_to_symbol_rate(..., offset=k)` picks sample `k` of each symbol
period; a timing error lets the neighbours' tails leak in.  A small
roll-off saves bandwidth but its slowly decaying tails make the link more
sensitive to timing errors.  (Fractional timing offsets and their
estimation are the subject of [11](11-frame-synchronization-and-timing.ipynb).)

In [ ]:
errors = np.linspace(-0.3, 0.3, 25)  # symbol periods
fig, ax = plt.subplots(figsize=(8, 4))
for beta in (0.05, 0.2, 0.5):
    t = ck.generate(
        C16,
        num_symbols=2**14,
        symbol_rate=1e9,
        sps=SPS,
        pulse=RRC(rolloff=beta),
        rng=4,
        device=DEVICE,
    )
    y = ck.filtering.matched_filter(t)
    snrs = [
        ck.metrics.snr(
            ck.multirate.decimate_to_symbol_rate(
                ck.timing.fft_fractional_delay(y, delay=e * SPS)
            )
        )
        for e in errors
    ]
    ax.plot(errors, snrs, label=f"β = {beta}")
ax.set_xlabel("Sampling error [symbol periods]")
ax.set_ylabel("Signal-to-ISI ratio [dB]")
ax.set_title("Noise-free 16-QAM: ISI from a sampling-phase error")
ax.legend()
plt.show()

## 6. How long must the pulse be?

An RRC pulse is infinitely long; `span` truncates it to that many symbol
periods.  Truncation leaves a residual ISI floor, the SNR a noise-free link
can reach.  Small roll-offs decay slowly and need long pulses: the default
`span=32` keeps the floor near 35 dB at β = 0.05 and above 50 dB at
β = 0.2, below the noise of any realistic link.

In [ ]:
for beta in (0.05, 0.2):
    row = []
    for span in (4, 8, 16, 32):
        t = ck.generate(
            C16,
            num_symbols=2**14,
            symbol_rate=1e9,
            sps=SPS,
            pulse=RRC(rolloff=beta, span=span),
            rng=5,
            device=DEVICE,
        )
        row.append(
            f"span {span:2d}: {ck.metrics.snr(ck.multirate.decimate_to_symbol_rate(ck.filtering.matched_filter(t))):5.1f} dB"
        )
    print(f"RRC β = {beta:4.2f}  ->  " + " | ".join(row))

**Summary**

* A Nyquist pulse is zero at all other symbol instants; the raised cosine
  does it in `(1 + β) Rs` of bandwidth.
* The matched filter `p*(-t)` maximizes the SNR; splitting the raised
  cosine into two RRCs gives both maximum SNR and zero ISI.
* Small roll-offs save bandwidth but cost timing tolerance and need longer
  pulses.

**Next:** [05 · Multirate processing and resampling](05-multirate-and-resampling.ipynb).